# Hent data på kandidaterne til folketingsvalget 2026

Denne notebook indeholder kode til at hente information på alle opstillede kandidater til folketingsvalget 2026 fra kombits sftp server.

Inden vi går i gang skal vi lige indlæse de nødvendige pakker. Vi bruger paramiko til at koble os på kombits server. Vi henter loginoplysningerne fra vores config-fil `config.py`.

In [76]:
import paramiko
import os
import re
import pandas as pd
import json

from config import HOST, PORT, USERNAME, PASSWORD, REMOTE_PATH

Forbind til sftp-serveren

In [77]:
transport = paramiko.Transport((HOST, PORT))
transport.connect(username=USERNAME, password=PASSWORD)
sftp = paramiko.SFTPClient.from_transport(transport)

Information om kandidaterne ligger i mappen `kandidat-data`. Gå ind i den mappe og download alle `.json` filerne.

In [ ]:
sftp.chdir("/data/folketingsvalg-135-24-03-2026/kandidat-data")

files = sftp.listdir()
for file in files:
    if file.endswith(".json"):
        print(f"Downloading {file}...")
        local_path = f"data/raw/kandidat-data/{file}"
        sftp.get(file, local_path)

Files in data folder:


Indlæs alle filerne og hent data på hver enkelt kandidat

In [ ]:
kandidat_data = []
for file in os.listdir("data/raw/kandidat-data"):
    if file.endswith(".json"):
        path = f"data/raw/kandidat-data/{file}"
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            for storkreds in data['IndenforParti']:
                for kandidater in storkreds['Kandidater']:
                    kandidat = {
                        'id': kandidater['Id'],
                        'navn': kandidater['Navn'],
                        'stemmeseddel_navn': kandidater['Stemmeseddelnavn'],
                        'parti': storkreds['Partinavn'],
                        'storkreds': data['Storkreds'],
                        "opstillingskredse": [opstillingskreds['Opstillingskreds'] for opstillingskreds in kandidater['Opstillingskredse']]
                    }
                    kandidat_data.append(kandidat)
                    
            for kandidater in data['UdenforParti']:
                kandidat = {
                    'id': kandidater['Id'],
                    'navn': kandidater['Navn'],
                    'stemmeseddel_navn': kandidater['Stemmeseddelnavn'],
                    'parti': "Udenfor parti",
                    'storkreds': data['Storkreds'],
                    "opstillingskredse": [opstillingskreds['Opstillingskreds'] for opstillingskreds in kandidater['Opstillingskredse']]
                }
                kandidat_data.append(kandidat)

Rens partinavnene så de passer til vores format.

In [ ]:
clean_parti_names = {
    "Danmarksdemokraterne - Inger Støjberg": "Danmarksdemokraterne",
    "SF - Socialistisk Folkeparti": "SF",
    "Det Konservative Folkeparti" : "Konservative",
    "Enhedslisten - De Rød-Grønne" : "Enhedslisten",
    "Radikale Venstre" : "Radikale",
    "Venstre, Danmarks Liberale Parti" : "Venstre",
    "Borgernes Parti - Lars Boje Mathiesen" : "Borgernes Parti"
}

for kandidat in kandidat_data:
    if kandidat['parti'] in clean_parti_names:
        kandidat['parti'] = clean_parti_names[kandidat['parti']]

Gem data som en `.csv`-fil

In [81]:
kandidat_df = pd.DataFrame(kandidat_data)
kandidat_df.to_csv("data/struktureret/kandidat_data.csv", index=False)